**Install a PDF text extraction library**

In [0]:
%pip install pypdf
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**List files and parse metadata from filenames**

In [0]:
"""
Phase 1: Bronze Ingestion
Extracts raw text from SEC 10-K PDFs stored in a Unity Catalog Volume,
parses company/fiscal_year metadata from filenames, and writes results
to a Delta table for downstream chunking + embedding.
"""

import os

volume_path = "/Volumes/rag_pipeline/main/raw_documents"
files = [f.path for f in dbutils.fs.ls(volume_path) if f.path.endswith(".pdf")]

# Parse company + fiscal_year from filename, e.g. GOOG_2025_10K.pdf
def parse_metadata(filepath):
    filename = os.path.basename(filepath)
    parts = filename.replace(".pdf", "").split("_")
    company = parts[0]
    fiscal_year = parts[1]
    return company, fiscal_year

for f in files:
    print(f, parse_metadata(f))

dbfs:/Volumes/rag_pipeline/main/raw_documents/AAPL_2023_10K.pdf ('AAPL', '2023')
dbfs:/Volumes/rag_pipeline/main/raw_documents/AAPL_2024_10K.pdf ('AAPL', '2024')
dbfs:/Volumes/rag_pipeline/main/raw_documents/AAPL_2025_10K.pdf ('AAPL', '2025')
dbfs:/Volumes/rag_pipeline/main/raw_documents/GOOG_2023_10K.pdf ('GOOG', '2023')
dbfs:/Volumes/rag_pipeline/main/raw_documents/GOOG_2024_10K.pdf ('GOOG', '2024')
dbfs:/Volumes/rag_pipeline/main/raw_documents/GOOG_2025_10K.pdf ('GOOG', '2025')
dbfs:/Volumes/rag_pipeline/main/raw_documents/META_2023_10K.pdf ('META', '2023')
dbfs:/Volumes/rag_pipeline/main/raw_documents/META_2024_10K.pdf ('META', '2024')
dbfs:/Volumes/rag_pipeline/main/raw_documents/META_2025_10K.pdf ('META', '2025')
dbfs:/Volumes/rag_pipeline/main/raw_documents/MSFT_2023_10K.pdf ('MSFT', '2023')
dbfs:/Volumes/rag_pipeline/main/raw_documents/MSFT_2024_10K.pdf ('MSFT', '2024')
dbfs:/Volumes/rag_pipeline/main/raw_documents/MSFT_2025_10K.pdf ('MSFT', '2025')
dbfs:/Volumes/rag_pipeline/m

**Extract text and build the Bronze DataFrame**

In [0]:
from pypdf import PdfReader
from pyspark.sql import Row

def extract_text(filepath):
    # dbutils.fs paths need /dbfs prefix removed for local file APIs on Volumes
    local_path = filepath.replace("dbfs:", "")
    reader = PdfReader(local_path)
    text = "\n".join([page.extract_text() or "" for page in reader.pages])
    return text

rows = []
for f in files:
    company, fiscal_year = parse_metadata(f)
    text = extract_text(f)
    rows.append(Row(
        file_path=f,
        company=company,
        fiscal_year=fiscal_year,
        raw_text=text,
        char_count=len(text)
    ))

bronze_df = spark.createDataFrame(rows)
display(bronze_df.select("company", "fiscal_year", "char_count"))

company,fiscal_year,char_count
AAPL,2023,209190
AAPL,2024,213379
AAPL,2025,215747
GOOG,2023,356753
GOOG,2024,369388
GOOG,2025,363171
META,2023,512512
META,2024,525407
META,2025,542302
MSFT,2023,368229


**Write to Bronze Delta table**

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("rag_pipeline.main.bronze_10k_filings")

display(spark.sql("SELECT company, fiscal_year, char_count FROM rag_pipeline.main.bronze_10k_filings ORDER BY company, fiscal_year"))

company,fiscal_year,char_count
AAPL,2023,209190
AAPL,2024,213379
AAPL,2025,215747
GOOG,2023,356753
GOOG,2024,369388
GOOG,2025,363171
META,2023,512512
META,2024,525407
META,2025,542302
MSFT,2023,368229
